# Session 7: Visualization for chemical measurements

A sorting-line operator at a plastics recycling plant reads a false-color map from a SWIR hyperspectral camera and decides which flakes go to the PET bale. PVC in a PET bale contaminates the melt and the reclaimer rejects the whole bale, whereas PET sent to the wrong bin goes to landfill. This notebook makes that map so that its colors keep the order of the numbers behind them. Section 1 works through a crop of a real SWIR survey. It reads the cube one band at a time, makes a band-ratio map, and shows the edges that a rainbow colormap draws and the contrast that a red and green composite puts in hue alone. Section 2 plans and makes the operator's figure for a test batch of known flakes on a belt, with a band pair assigned to you. Section 3 asks which pixels make a flake's spectrum when some flakes catch the lamp's glare. Section 4 is the judgment cell.

Dataset A is real: M. Balsi, M. Moroni, and S. Bouchelaghem, "SWIR hyperspectral data cubes for plastics detection in the environment", Mendeley Data, version 1, 2025, [doi:10.17632/25hfz5c8dt.1](https://doi.org/10.17632/25hfz5c8dt.1), CC BY 4.0, accessed 2026-10-01, described in M. Balsi, M. Moroni, and S. Bouchelaghem, *Remote Sensing* 17 (2025) 938, [doi:10.3390/rs17050938](https://doi.org/10.3390/rs17050938). The files in `data/apr22/` are a crop of the April 2022 survey's 81 band images and its PVC mask, saved as PNG; `data/apr22_spectra.csv` holds mean spectra over the whole survey. Dataset B, in `data/batch/`, is simulated from those spectra; `data/README.md` describes both.

**To submit.** Put your name in the next cell. Click **Restart** in the notebook toolbar, then **Run All**, check that every cell ran without an error, save, and upload `activity.ipynb` to the Session 7 assignment on Canvas by 1:30 pm on Tuesday, October 27. Submissions are individual. The test batch and every band pair are in the repository, so the graders' run reads the same data yours did. The notebook is scored 0 to 3: 1 if it runs from top to bottom after a restart, and 0 to 2 for the judgment cell, by the rubric in the syllabus, with the format in `ai-practices.md`.

*Name:*



## Setup and imports

The next cell imports `sys`, `Path` from `pathlib`, NumPy, pandas, Matplotlib, and `imread` from `matplotlib.image`. An image file holds one number per pixel for each color channel. `imread` is new this session, and it reads an image file into a NumPy array, a 2-D array for a single-channel image such as one band of a hyperspectral camera. The cell prints nothing when it works.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.image import imread

## Parameters

The first block sets Dataset A's folder and spectra table, its band pair (1200 nm, where PVC absorbs more than grass does, over 1250 nm, where it does not), the level below which a band carries no signal, the percentiles that set color limits, the three bands of the composite, and the weights that turn a color into the gray a grayscale print shows. The second block sets Dataset B's folder, your band pair, the four polymers, the pixel size and scale bar, and the band and level that section 3 uses to find glints. Leave `NUMERATOR_NM` and `DENOMINATOR_NM` as `CHANGE-ME` until section 2.

In [ ]:
CROP_DIR = Path("data/apr22")
SPECTRA_CSV = Path("data/apr22_spectra.csv")
A_NUMERATOR_NM = 1200                 # PVC absorbs here more than grass and PET do
A_DENOMINATOR_NM = 1250               # and not here, so the ratio cancels brightness
EMPTY_DN = 10                         # a band whose largest value is below this carries no signal, DN
LIMIT_PERCENTILES = (1, 99)           # color limits, percentiles of the pixels shown
COMPOSITE_NM = (1550, 1200, 1250)     # the composite's red, green, and blue bands, nm
LUMINANCE = (0.2126, 0.7152, 0.0722)  # how much red, green, and blue light each add to brightness, sRGB

BATCH_DIR = Path("data/batch")
NUMERATOR_NM = "CHANGE-ME"            # your pair's numerator, nm, from data/band_pairs.csv, section 2
DENOMINATOR_NM = "CHANGE-ME"          # your pair's denominator, nm, section 2
POLYMERS = ["PET", "PVC", "PE", "PP"]
PIXEL_MM = 2.0                        # belt per pixel, mm
SCALE_BAR_MM = 50                     # the map's scale bar, mm
GLINT_BAND_NM = 1100                  # section 3: the band where glints are found
SATURATION_DN = 250                   # section 3: treatment B drops pixels at or above this, DN

## 1. Explore: reading a cube one band at a time

A hyperspectral camera records one image per wavelength band, so the survey arrives as 81 image files, 900 to 1700 nm in steps of 10 nm, and the spectrum of one pixel is that pixel's value read from every file in turn. The next cell lists the files, sorts them so that name order is wavelength order, and reads the first. `imread` returns an 8-bit PNG as fractions of 255, so `read_band` multiplies by 255 to give back the camera's digital numbers (DN), raw signal that is not yet reflectance.

In [ ]:
def read_band(path):
    """One band image as a 2-D array of digital numbers, 0 to 255."""
    return imread(path) * 255


def crop_band(nm):
    """One band of Dataset A by its wavelength in nm."""
    return read_band(CROP_DIR / f"sequence_1_SWIR_fl01500_1659_{nm:04d}.png")


crop_paths = sorted(CROP_DIR.glob("sequence_*.png"))
crop_nm = np.array([int(path.stem[-4:]) for path in crop_paths])   # nm, the last four characters of each name
first = read_band(crop_paths[0])
print(len(crop_paths), "bands,", crop_nm[0], "to", crop_nm[-1], "nm")
print(f"{crop_nm[0]} nm: {first.shape[0]} by {first.shape[1]} pixels, values {first.min():.0f} to {first.max():.0f} DN")

The first band is zero everywhere. The deposit says that 900 to 930 nm are empty, and the next cell checks every band, printing those whose largest value is under `EMPTY_DN`. It then counts the pixels that are zero at 1250 nm. The survey is a mosaic of flight lines, and the area outside it is filled with zeros that were never measured. A zero there means no data, so before any ratio those pixels become `np.nan`, NumPy's value for "not a number". Matplotlib leaves `np.nan` blank.

In [ ]:
for path, nm in zip(crop_paths, crop_nm):
    peak = read_band(path).max()
    if peak < EMPTY_DN:
        print(f"{nm} nm: largest value {peak:.0f} DN")
outside = crop_band(A_DENOMINATOR_NM) == 0
print(f"{outside.mean():.1%} of pixels are 0 at {A_DENOMINATOR_NM} nm, outside the mosaic")

To compare pixels that catch different amounts of light, divide one band by another. The ratio of 1200 nm, where PVC absorbs, over 1250 nm, where it does not, cancels how bright a pixel is and keeps how deep the absorption is. `np.where(valid, a, b)` takes `a` where `valid` is true and `b` elsewhere. Here it blanks the zeros and keeps the division from meeting one. The map is drawn with no color limits, so `imshow` sets them at the minimum and maximum.

In [ ]:
numerator = crop_band(A_NUMERATOR_NM)
denominator = crop_band(A_DENOMINATOR_NM)
valid = (numerator > 0) & (denominator > 0)
ratio_A = np.where(valid, numerator / np.where(valid, denominator, 1), np.nan)
low, high = np.percentile(ratio_A[valid], LIMIT_PERCENTILES)
print(f"ratio {ratio_A[valid].min():.3f} to {ratio_A[valid].max():.3f}; "
      f"{LIMIT_PERCENTILES[0]}st and {LIMIT_PERCENTILES[1]}th percentiles {low:.3f} and {high:.3f}")

fig, ax = plt.subplots(figsize=(9, 3.5))
image = ax.imshow(ratio_A, cmap="viridis")
fig.colorbar(image, ax=ax, label=f"ratio, {A_NUMERATOR_NM} nm over {A_DENOMINATOR_NM} nm")
ax.set_xlabel("pixel")
ax.set_ylabel("pixel")
ax.set_title("April 2022 survey, crop, limits at minimum and maximum")
fig.tight_layout()
plt.show()

The map shows the crop, 1085 pixels across and 400 down, in one dark purple, with the area outside the mosaic blank at the right. The colorbar runs from 0.067 to 19.0, set by a few pixels at the mosaic's edge, whereas 98% of the pixels lie between the 1st and 99th percentiles, 0.739 and 0.981, so the whole survey sits in the bottom 5% of the colors. The next cell draws the map with limits at those percentiles in two colormaps, jet and viridis. Jet is a rainbow. Viridis is perceptually uniform: equal steps in ratio look like equal steps in color. The cell then redraws both maps as a grayscale print shows them.

In [ ]:
def as_gray(values, cmap, low, high):
    """How light each pixel's color looks, 0 for black to 1 for white (CIELAB lightness over 100)."""
    rgb = plt.colormaps[cmap]((values - low) / (high - low))
    linear = np.where(rgb <= 0.04045, rgb / 12.92, ((rgb + 0.055) / 1.055) ** 2.4)   # undo the screen's gamma
    y = LUMINANCE[0] * linear[:, :, 0] + LUMINANCE[1] * linear[:, :, 1] + LUMINANCE[2] * linear[:, :, 2]
    lightness = np.where(y > 0.008856, 1.16 * np.cbrt(y) - 0.16, 9.033 * y)   # the CIELAB formula, sRGB standard
    return np.where(np.isnan(values), np.nan, lightness)


fig, axes = plt.subplots(2, 2, figsize=(12, 5.5))
for column, cmap in enumerate(["jet", "viridis"]):
    image = axes[0, column].imshow(ratio_A, cmap=cmap, vmin=low, vmax=high)
    fig.colorbar(image, ax=axes[0, column], label="ratio")
    axes[0, column].set_title(f"{cmap}, limits at percentiles")
    image = axes[1, column].imshow(as_gray(ratio_A, cmap, low, high), cmap="gray", vmin=0, vmax=1)
    fig.colorbar(image, ax=axes[1, column], label="gray level")
    axes[1, column].set_title(f"{cmap}, as a grayscale print")
for ax in axes.flat:
    ax.set_xlabel("pixel")
    ax.set_ylabel("pixel")
fig.tight_layout()
plt.show()

pvc_mask = imread(CROP_DIR / "03_PVC.png") > 0.5
print(f"median ratio inside the PVC mask {np.median(ratio_A[pvc_mask & valid]):.3f}, "
      f"outside it {np.median(ratio_A[~pvc_mask & valid]):.3f}")

In the top row the PVC object is the low-ratio region at the upper left, dark blue in jet and dark purple in viridis, and grass fills the rest. In jet's grayscale print a white ring surrounds the object, and the object's core and the brightest grass print almost equally dark, whereas in viridis's print the object is darkest and the grass lightest, with no ring. The median ratio is 0.802 inside the PVC mask and 0.940 outside it. A sorting display often shows three bands as red, green, and blue. The next cell does so for 1550, 1200, and 1250 nm, then draws each band alone.

In [ ]:
channels = []
for nm in COMPOSITE_NM:
    band = crop_band(nm)
    lo, hi = np.percentile(band[valid], LIMIT_PERCENTILES)
    channels.append(np.clip((band - lo) / (hi - lo), 0, 1))   # 0 at the band's 1st percentile, 1 at its 99th

fig, axes = plt.subplots(2, 2, figsize=(12, 5.5))
axes[0, 0].imshow(np.dstack(channels))
axes[0, 0].set_title(f"composite: red {COMPOSITE_NM[0]}, green {COMPOSITE_NM[1]}, blue {COMPOSITE_NM[2]} nm")
for ax, nm, channel in zip(axes.flat[1:], COMPOSITE_NM, channels):
    image = ax.imshow(channel, cmap="gray", vmin=0, vmax=1)
    fig.colorbar(image, ax=ax, label="fraction of percentile range")
    ax.set_title(f"{nm} nm alone")
for ax in axes.flat:
    ax.set_xlabel("pixel")
    ax.set_ylabel("pixel")
fig.tight_layout()
plt.show()

The composite draws the PVC object magenta, the grass teal at nearly the same lightness, and the area outside the mosaic black. Its contrast is mostly hue, red against green. A simulated view for the most common color vision deficiency keeps about a third of that contrast, and a grayscale print keeps almost none. The three panels say what the colors carry: PVC's median is 90 DN at 1550 nm against 74 elsewhere, 91 against 105 at 1200 nm, and 119 against 112 at 1250 nm. The last figure plots the mean spectrum of each polymer and of grass over the whole survey. Each series has its own line style and marker, and the two ratio bands are marked.

In [ ]:
spectra = pd.read_csv(SPECTRA_CSV)
styles = {"grass": "-", "PET": "--o", "PE": "-.s", "PVC": ":^", "PP": "--v", "PS": "-.D"}

fig, ax = plt.subplots(figsize=(9, 4.5))
for name, style in styles.items():
    ax.plot(spectra["wavelength_nm"], spectra[name], style, color="black", fillstyle="none", markevery=5, label=name)
for nm in (A_NUMERATOR_NM, A_DENOMINATOR_NM):
    ax.axvline(nm, color="gray", linestyle="-.", linewidth=1)
ax.set_xlabel("wavelength (nm)")
ax.set_ylabel("mean signal over object pixels (DN)")
ax.set_title("April 2022 survey, whole mosaic, mean spectra")
ax.legend(ncol=2)
fig.tight_layout()
plt.show()
spectra[spectra["wavelength_nm"].isin([A_NUMERATOR_NM, A_DENOMINATOR_NM])]

From 950 to 1700 nm every series rises to a broad maximum near 1080 nm, then dips at 1130 nm and again from 1350 to 1450 nm, where water in the air between the drone and the ground absorbs. Outside those dips, grass lies below every plastic except PVC at 1200 nm. At 1200 nm PVC reads 100.0 DN against 134.1 DN at 1250 nm, a ratio of 0.746, whereas PET's two values give 0.953 and grass's 0.950. One ratio therefore separates PVC from grass and leaves PET where grass is.

**Explain.** In the grayscale view of the jet map, the PVC object is ringed by a light band that the viridis view does not have. In two sentences, say what the ratio does across that ring, from the colorbars, and what in jet's run of colors makes a smooth change look like an edge.

*Your answer:*

## 2. Apply: making the operator's figure for a test batch

A plant checks its display by running known flakes past the camera. Your test batch, in `data/batch/`, is 76 band images, 950 to 1700 nm, of PET, PVC, PE, and PP flakes on a dark belt, each 200 by 400 pixels at 2 mm per pixel. One mask per polymer marks its flakes, drawn generously as the survey's masks are. It is simulated from the survey's measured spectra. Each flake has its own brightness and thickness, and about half catch a glint of the lamp. `data/README.md` says what is measured and what is modeled. Each of you has one band pair from `data/band_pairs.csv`, listed on the board.

**[TASK 1]** Set `NUMERATOR_NM` and `DENOMINATOR_NM` in the Parameters cell to your pair's two wavelengths, as numbers, run that cell again, then run the next cell. It reads your two bands and the four masks, makes `ratio_map` with the belt left blank, and reads every band to build `flake_spectra`, the mean spectrum of PET, PVC, and the belt over their masks. Until the pair is set, the cell stops with `FileNotFoundError` for `data/batch/test_batch_CHANGE-ME.png`.

In [ ]:
def batch_band(nm):
    """One band of the test batch by its wavelength in nm."""
    return read_band(BATCH_DIR / f"test_batch_{nm:0>4}.png")


batch_num = batch_band(NUMERATOR_NM)
batch_den = batch_band(DENOMINATOR_NM)
masks = {polymer: imread(BATCH_DIR / f"{polymer}.png") > 0.5 for polymer in POLYMERS}
on_flake = masks["PET"] | masks["PVC"] | masks["PE"] | masks["PP"]
ratio_map = np.where(on_flake, batch_num / np.where(on_flake, batch_den, 1), np.nan)   # belt pixels left blank

batch_paths = sorted(BATCH_DIR.glob("test_batch_*.png"))
wavelength_nm = np.array([int(path.stem[-4:]) for path in batch_paths])
flake_spectra = {"PET": [], "PVC": [], "belt": []}
for path in batch_paths:
    band = read_band(path)
    flake_spectra["PET"].append(band[masks["PET"]].mean())
    flake_spectra["PVC"].append(band[masks["PVC"]].mean())
    flake_spectra["belt"].append(band[~on_flake].mean())

on_flake_ratios = ratio_map[on_flake]
print(f"{on_flake.sum()} pixels on flakes of {on_flake.size}; {len(wavelength_nm)} bands read")
print(f"ratio on flakes {on_flake_ratios.min():.3f} to {on_flake_ratios.max():.3f}; "
      f"{LIMIT_PERCENTILES[0]}st and {LIMIT_PERCENTILES[1]}th percentiles "
      f"{np.percentile(on_flake_ratios, LIMIT_PERCENTILES[0]):.3f} and {np.percentile(on_flake_ratios, LIMIT_PERCENTILES[1]):.3f}")

**Ask the assistant** before TASK 2, in Claude Code, with today's AI practice, *explore and plan before code*. Type `/plan` followed by the request below. Claude Code then writes a plan, a Markdown document you can comment on, and writes no code until you approve it. If `/plan` is not in the menu that opens when you type `/`, click the mode indicator at the bottom of the prompt box and choose **Plan**. If neither appears, send the request without `/plan`.

`/plan A two-panel figure for the sorting-line operator. Left: ratio_map, my band NUMERATOR_NM over DENOMINATOR_NM for the test batch. Right: the mean spectrum of PET, PVC, and the belt from flake_spectra against wavelength_nm. Say in words which colormap and color limits you would use and why, how the series are told apart, and what each axis and the colorbar are labeled. No code until I approve.`

Read the plan against the last cell's output and section 1 before you approve it. Are its limits the minimum and maximum, or percentiles? Does its colormap's grayscale rise evenly? Are the series told apart without color? Write your changes as comments on the plan document, or type them in the prompt box as a reply. You do not need to approve the plan, because the TASK 2 line is yours to write from it. What the plan proposed and what you did with it, whether you changed it or verified it against your output, is one candidate for part (b) of the judgment cell.

**[TASK 2]** Make the figure your edited plan describes. The cell draws both panels, the scale bar and every label; the one line left to you sets the map's colormap and color limits. Until it is filled, the cell stops with `ValueError: Ellipsis is not a valid value for cmap`.

In [ ]:
fig, (ax_map, ax_spec) = plt.subplots(1, 2, figsize=(13, 4.5))
image = ax_map.imshow(ratio_map,
                      cmap=..., vmin=..., vmax=...)   # [TASK 2] the colormap and color limits of your edited plan
fig.colorbar(image, ax=ax_map, label=f"ratio, {NUMERATOR_NM} nm over {DENOMINATOR_NM} nm")
bar_row = ratio_map.shape[0] - 8                # pixels; the scale bar sits near the bottom left
ax_map.plot([8, 8 + SCALE_BAR_MM / PIXEL_MM], [bar_row, bar_row], color="black", linewidth=3)
ax_map.text(8, bar_row - 4, f"{SCALE_BAR_MM} mm")
ax_map.set_xlabel("pixel across the belt")
ax_map.set_ylabel("pixel along the belt")
ax_map.set_title("Test batch, belt left blank")
for name, style in [("PET", "-o"), ("PVC", "--s"), ("belt", ":^")]:
    ax_spec.plot(wavelength_nm, flake_spectra[name], style, color="black", fillstyle="none", markevery=5, label=name)
for nm in (NUMERATOR_NM, DENOMINATOR_NM):
    ax_spec.axvline(nm, color="gray", linestyle="-.", linewidth=1)
ax_spec.set_xlabel("wavelength (nm)")
ax_spec.set_ylabel("mean signal over the mask (DN)")
ax_spec.set_title("Test batch, mean spectra")
ax_spec.legend()
fig.tight_layout()
plt.show()

*Write your one sentence description of the plot here.*

The next cell shows what jet would have done to your map. It redraws `ratio_map` in jet and in viridis as a grayscale print shows them, with limits at the 1st and 99th percentiles of the flake pixels. For each colormap it prints the gray level of the PET and PVC medians, then the gray levels at the 25th and 75th percentiles of each polymer's pixels, the range that the middle half of its pixels prints over. The gray level is the lightness a color appears to have, 0 for black and 1 for white, and two that differ by less than 0.05 look alike in print. jet can fail in two ways: it can print PET and PVC alike, or it can spread one polymer's middle half over a much wider range of grays than viridis does, so that thick and thin flakes of one polymer look like two materials.

In [ ]:
low_b, high_b = np.percentile(on_flake_ratios, LIMIT_PERCENTILES)
pet_median = np.median(ratio_map[masks["PET"]])
pvc_median = np.median(ratio_map[masks["PVC"]])

fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
for ax, cmap in zip(axes, ["jet", "viridis"]):
    image = ax.imshow(as_gray(ratio_map, cmap, low_b, high_b), cmap="gray", vmin=0, vmax=1)
    fig.colorbar(image, ax=ax, label="gray level")
    ax.set_title(f"{cmap}, as a grayscale print")
    ax.set_xlabel("pixel across the belt")
    ax.set_ylabel("pixel along the belt")
    gray_pet, gray_pvc = as_gray(np.array([[pet_median, pvc_median]]), cmap, low_b, high_b)[0]
    print(f"{cmap}: PET median {pet_median:.3f} prints at gray {gray_pet:.2f}, PVC median {pvc_median:.3f} at {gray_pvc:.2f}")
    for polymer in ["PET", "PVC"]:
        gray = as_gray(np.where(masks[polymer], ratio_map, np.nan), cmap, low_b, high_b)[masks[polymer]]
        q1, q3 = np.percentile(gray, [25, 75])
        print(f"    {polymer}: middle half of its pixels prints from gray {q1:.2f} to {q3:.2f}")
fig.tight_layout()
plt.show()

*Write your one sentence description of the plot here.*

**[TASK 3]** Put numbers on the map. The cell prints each polymer's median ratio over its mask, and the belt's; the line that picks a polymer's pixels from `ratio_map` with its mask is left to you. Until it is filled, the cell stops with `TypeError: unsupported operand type(s) for /: 'ellipsis' and 'int'`.

In [ ]:
rows = []
for polymer in POLYMERS:
    pixels = ...   # [TASK 3] this polymer's ratios: ratio_map at the pixels of masks[polymer]
    rows.append({"polymer": polymer, "median_ratio": np.median(pixels), "pixels": pixels.size})
belt = ~on_flake & (batch_den > 0)
rows.append({"polymer": "belt", "median_ratio": np.median(batch_num[belt] / batch_den[belt]), "pixels": int(belt.sum())})
medians = pd.DataFrame(rows)
medians.round(3)

**Explain.** Before you compare, write two sentences. Say which polymers your figure lets the operator tell apart and which it does not, citing your medians. Then say which of jet's two failures your pair shows in a grayscale print, PET and PVC printed alike or one polymer spread over a wider range of grays than viridis gives it, citing the gray levels printed above. Then compare with a neighbor: set your band pairs, your PET and PVC medians, and what jet did side by side. Add one sentence on where your two answers differ and what in your band pair explains it.

*Your answer:*

## 3. Analyze: choosing the pixels that make a flake's spectrum

About half the flakes caught a glint, a spot where the lamp reflects straight off the surface. A glint is so bright that the camera clips it at 255 in most bands, so its ratio is near 1 whatever the polymer. The masks include the glints. Two treatments of a polymer's spectrum are defensible. Treatment A averages every pixel the mask assigns to the polymer, with no threshold to choose. Treatment B first drops pixels at or above `SATURATION_DN` at `GLINT_BAND_NM`, which removes the glints at the cost of a threshold. The cell counts the pixels B drops. Under each treatment, it then prints the ratio of PET's and PVC's mean spectra at your pair, and the PVC minus PET difference.

In [ ]:
unsaturated = batch_band(GLINT_BAND_NM) < SATURATION_DN
treatments = {"A, every masked pixel": np.ones_like(on_flake), "B, glints dropped": unsaturated}
spectrum_ratio = {}
for polymer in ["PET", "PVC"]:
    print(f"{polymer}: {masks[polymer].sum()} masked pixels, {(masks[polymer] & ~unsaturated).sum()} at or above {SATURATION_DN} DN")
for label, keep in treatments.items():
    for polymer in ["PET", "PVC"]:
        pixels = masks[polymer] & keep
        spectrum_ratio[(label, polymer)] = batch_num[pixels].mean() / batch_den[pixels].mean()
    difference = spectrum_ratio[(label, "PVC")] - spectrum_ratio[(label, "PET")]
    print(f"treatment {label}: PET {spectrum_ratio[(label, 'PET')]:.3f}, PVC {spectrum_ratio[(label, 'PVC')]:.3f}, "
          f"PVC minus PET {difference:+.3f}")

**Explain.** Give PET's and PVC's spectrum ratios and their difference under each treatment, the number of pixels treatment B dropped from each polymer, and the fraction by which the glints change the PET to PVC difference, as a fraction of treatment B's difference. This is the evidence part (a) of the judgment cell needs.

*Your answer:*

## 4. Judgment cell

The cell below is scored 0 to 2 by the rubric in the syllabus, in the format that `ai-practices.md` gives, and you write it yourself. Part (a): would your figure let the operator tell PET from PVC, and which treatment would you build the display's spectra on, and why? Cite your PET, PVC, and belt medians from TASK 3 and the PVC minus PET difference under both treatments, and say what a PVC flake read as PET would mean for the bale. Part (b): one judgment entry, a suggestion from the assistant that you changed, rejected, or verified, and the reason from your data. The plan before TASK 2 is the natural source. The reason has to cite a value from your output, such as the printed range against the percentiles or a gray level, not a general rule. This session's AI practice, named in the notes, is *explore and plan before code*. An entry that came from using it is the expected one, although any entry grounded in your data counts.

**Judgment cell**

**(a)**

**(b) Judgment entry**

- **What the assistant suggested:**
- **What I did:**
- **Why, from my data:** 

## Software environment

The last cell prints the Python and package versions this notebook ran with, so a result can always be traced to the software that produced it.

In [ ]:
import matplotlib

print(f"Python     {sys.version.split()[0]}")
print(f"NumPy      {np.__version__}")
print(f"pandas     {pd.__version__}")
print(f"Matplotlib {matplotlib.__version__}")